<a href="https://colab.research.google.com/github/SattamAltwaim/KAUST-Mawhiba-IOAI-2026/blob/main/competition/baseline_notebook.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>&nbsp;&nbsp;<a href="https://www.kaggle.com/competitions/kaust-mawhiba-ioai-competition-1" target="_blank"><img src="https://img.shields.io/badge/Kaggle-Join%20Competition-20BEFF?logo=kaggle&logoColor=white" alt="Join Competition"/></a>

# ML Challenge — Baseline (Logistic + Linear Regression)

This notebook provides a minimal baseline using only:
- **Logistic Regression** for the classification task (Churn)
- **Linear Regression** for the regression task (Insurance Charges)

No feature engineering, no sampling, no hyperparameter tuning.

$$\text{Score} = \frac{\text{Macro F1} + \max(0,\; R^2)}{2} \times 100$$

**Expected baseline score: ~72**. Your job is to beat this.

In [1]:
!pip install -q kagglehub

In [2]:
import kagglehub
import pandas as pd
import numpy as np
import os

# ════════════════════════════════════════════════════════════
#  DATA LOADING — do not modify
# ════════════════════════════════════════════════════════════

DATASET_SLUG = "sattamjaltwaim/kaust-mawhiba-ioai-competition-1"  # ← replace with actual slug
data_path = kagglehub.dataset_download(DATASET_SLUG)

train_clf = pd.read_csv(os.path.join(data_path, "train_clf.csv"), index_col="id")
test_clf  = pd.read_csv(os.path.join(data_path, "test_clf.csv"),  index_col="id")
train_reg = pd.read_csv(os.path.join(data_path, "train_reg.csv"), index_col="id")
test_reg  = pd.read_csv(os.path.join(data_path, "test_reg.csv"),  index_col="id")

print(f"Classification — Train: {train_clf.shape}, Test: {test_clf.shape}")
print(f"Regression     — Train: {train_reg.shape}, Test: {test_reg.shape}")
print(f"\nChurn rate: {train_clf['Churn'].mean():.1%} — this is imbalanced!")

100%|██████████| 148k/148k [00:00<00:00, 565kB/s]

Extracting files...
Classification — Train: (5625, 20), Test: (1407, 19)
Regression     — Train: (1070, 7), Test: (268, 6)

Churn rate: 26.6% — this is imbalanced!


---

## Quick EDA

In [3]:
print("=" * 50)
print("CLASSIFICATION DATASET")
print("=" * 50)
print(f"\nTarget distribution:")
print(train_clf["Churn"].value_counts())
print(f"\nFeature types:")
print(train_clf.dtypes.value_counts())
print(f"\nSample rows:")
train_clf.head(3)

CLASSIFICATION DATASET

Target distribution:
Churn
0    4130
1    1495
Name: count, dtype: int64

Feature types:
object     15
int64       3
float64     2
Name: count, dtype: int64

Sample rows:


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
id,,,,,,,,,,,,,,,,,,,,
0,Male,0,Yes,Yes,65,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,No,No,Two year,No,Credit card (automatic),94.55,6078.75,0
1,Male,0,No,No,26,No,No phone service,DSL,No,No,Yes,Yes,No,No,Month-to-month,No,Electronic check,35.75,1022.50,0
2,Female,0,Yes,No,68,Yes,Yes,Fiber optic,No,Yes,Yes,Yes,No,No,Two year,No,Credit card (automatic),90.20,6297.65,0


In [4]:
print("=" * 50)
print("REGRESSION DATASET")
print("=" * 50)
print(f"\nTarget statistics:")
print(train_reg["charges"].describe())
print(f"\nFeature types:")
print(train_reg.dtypes.value_counts())
print(f"\nSample rows:")
train_reg.head(3)

REGRESSION DATASET

Target statistics:
count     1070.000000
mean     13346.089736
std      12019.510778
min       1121.873900
25%       4897.667387
50%       9575.442100
75%      16746.657400
max      62592.873090
Name: charges, dtype: float64

Feature types:
object     3
int64      2
float64    2
Name: count, dtype: int64

Sample rows:


,age,sex,bmi,children,smoker,region,charges
id,,,,,,,
0,46,female,19.95,2,no,northwest,9193.83850
1,47,female,24.32,0,no,northeast,8534.67180
2,52,female,24.86,0,no,southeast,27117.99378


---

## Task 1: Classification — Logistic Regression Baseline

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score, classification_report

X_train_clf = train_clf.drop("Churn", axis=1).copy()
y_train_clf = train_clf["Churn"].copy()
X_test_clf = test_clf.copy()

# Encode every object column with LabelEncoder
for col in X_train_clf.select_dtypes(include="object").columns:
    le = LabelEncoder()
    combined = pd.concat([X_train_clf[col], X_test_clf[col]]).astype(str)
    le.fit(combined)
    X_train_clf[col] = le.transform(X_train_clf[col].astype(str))
    X_test_clf[col] = le.transform(X_test_clf[col].astype(str))

clf_model = LogisticRegression(max_iter=1000, random_state=42)
clf_model.fit(X_train_clf, y_train_clf)

# Cross-val estimate
cv_scores = cross_val_score(clf_model, X_train_clf, y_train_clf, cv=5, scoring="f1_macro")
print(f"5-Fold CV Macro F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Full training set report
y_train_pred_clf = clf_model.predict(X_train_clf)
print(f"\nTraining Macro F1: {f1_score(y_train_clf, y_train_pred_clf, average='macro'):.4f}")
print(classification_report(y_train_clf, y_train_pred_clf, target_names=["No Churn", "Churn"]))

y_pred_clf = clf_model.predict(X_test_clf)
print(f"Test predictions: {len(y_pred_clf)} samples, predicted churn rate: {y_pred_clf.mean():.1%}")

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please a

5-Fold CV Macro F1: 0.7273 (+/- 0.0196)

Training Macro F1: 0.7355
              precision    recall  f1-score   support

    No Churn       0.85      0.90      0.87      4130
       Churn       0.66      0.55      0.60      1495

    accuracy                           0.80      5625
   macro avg       0.75      0.72      0.74      5625
weighted avg       0.80      0.80      0.80      5625

Test predictions: 1407 samples, predicted churn rate: 24.2%


---

## Task 2: Regression — Linear Regression Baseline

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

X_train_reg = train_reg.drop("charges", axis=1).copy()
y_train_reg = train_reg["charges"].copy()
X_test_reg = test_reg.copy()

# Encode every object column with LabelEncoder
for col in X_train_reg.select_dtypes(include="object").columns:
    le = LabelEncoder()
    combined = pd.concat([X_train_reg[col], X_test_reg[col]]).astype(str)
    le.fit(combined)
    X_train_reg[col] = le.transform(X_train_reg[col].astype(str))
    X_test_reg[col] = le.transform(X_test_reg[col].astype(str))

reg_model = LinearRegression()
reg_model.fit(X_train_reg, y_train_reg)

# Cross-val estimate
cv_r2 = cross_val_score(reg_model, X_train_reg, y_train_reg, cv=5, scoring="r2")
print(f"5-Fold CV R²: {cv_r2.mean():.4f} (+/- {cv_r2.std():.4f})")

# Full training set report
y_train_pred_reg = reg_model.predict(X_train_reg)
train_r2 = r2_score(y_train_reg, y_train_pred_reg)
train_mae = mean_absolute_error(y_train_reg, y_train_pred_reg)
print(f"\nTraining R²:  {train_r2:.4f}")
print(f"Training MAE: ${train_mae:,.0f}")

y_pred_reg = reg_model.predict(X_test_reg)
print(f"\nTest predictions: {len(y_pred_reg)} samples, predicted mean charge: ${y_pred_reg.mean():,.0f}")

5-Fold CV R²: 0.7339 (+/- 0.0487)

Training R²:  0.7417
Training MAE: $4,209

Test predictions: 268 samples, predicted mean charge: $13,193


---

## Baseline Score Estimate

In [7]:
train_f1 = f1_score(y_train_clf, y_train_pred_clf, average="macro")
train_r2 = r2_score(y_train_reg, y_train_pred_reg)
baseline_score = (train_f1 + max(0, train_r2)) / 2 * 100

print("=" * 50)
print(f"BASELINE SCORE ESTIMATE: {baseline_score:.1f} / 100")
print("=" * 50)
print(f"  Macro F1 (clf): {train_f1:.4f}")
print(f"  R² (reg):       {train_r2:.4f}")
print()
print("Ideas to improve:")
print("  Classification:")
print("    - Handle imbalance: class_weight='balanced', threshold tuning, stratified sampling")
print("    - One-hot encode instead of label encode (for non-ordinal features)")
print("    - Feature engineering: tenure bins, service bundles, charge ratios")
print("    - Try: RandomForest, XGBoost, LightGBM")
print("  Regression:")
print("    - Key insight: smoker x BMI interaction is highly predictive")
print("    - Polynomial features, age bins, BMI categories")
print("    - Log-transform the target (charges are right-skewed)")
print("    - Try: RandomForest, GradientBoosting, XGBoost")

BASELINE SCORE ESTIMATE: 73.9 / 100
  Macro F1 (clf): 0.7355
  R² (reg):       0.7417

Ideas to improve:
  Classification:
    - Handle imbalance: SMOTE, class_weight='balanced', threshold tuning
    - One-hot encode instead of label encode (for non-ordinal features)
    - Feature engineering: tenure bins, service bundles, charge ratios
    - Try: RandomForest, XGBoost, LightGBM
  Regression:
    - Key insight: smoker x BMI interaction is highly predictive
    - Polynomial features, age bins, BMI categories
    - Log-transform the target (charges are right-skewed)
    - Try: RandomForest, GradientBoosting, XGBoost


---

## Generate Submission

Pass your predictions to the function below. **Do not modify it.**

- `y_pred_clf` — array of `0` or `1`, length = `len(test_clf)`
- `y_pred_reg` — array of floats, length = `len(test_reg)`

In [8]:
# ════════════════════════════════════════════════════════════
#  SUBMISSION GENERATOR — do not modify
# ════════════════════════════════════════════════════════════

def generate_submission(y_pred_clf, y_pred_reg, filename="submission.csv"):
    y_clf = np.asarray(y_pred_clf, dtype=int)
    y_reg = np.asarray(y_pred_reg, dtype=float)

    assert len(y_clf) == len(test_clf), (
        f"Classification predictions length {len(y_clf)} != test size {len(test_clf)}"
    )
    assert len(y_reg) == len(test_reg), (
        f"Regression predictions length {len(y_reg)} != test size {len(test_reg)}"
    )

    clf_ids = [f"clf_{i}" for i in range(len(y_clf))]
    reg_ids = [f"reg_{i}" for i in range(len(y_reg))]

    submission = pd.DataFrame({
        "id": clf_ids + reg_ids,
        "prediction": list(y_clf.astype(float)) + list(y_reg),
    })
    submission.to_csv(filename, index=False)
    print(f"Saved {filename}  ({len(submission)} rows)")
    print(f"  Classification: {len(y_clf)} predictions  (predicted churn rate: {y_clf.mean():.1%})")
    print(f"  Regression:     {len(y_reg)} predictions  (predicted mean charge: ${y_reg.mean():,.0f})")


generate_submission(y_pred_clf, y_pred_reg)

Saved submission.csv  (1675 rows)
  Classification: 1407 predictions  (predicted churn rate: 24.2%)
  Regression:     268 predictions  (predicted mean charge: $13,193)
